<a href="https://colab.research.google.com/github/PyDev2069/Claude-With-AWS-Bedrock/blob/main/Claude_Coaching_AWS_Bedrock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.7 MB/s eta 0:00:00


First import the modules which are going to be needed for the course.
**bold text**

Here we have taken os for handling secrets.
boto3 for connecting the notebook with the AWS Bedrock
and finally userdata to access secrets securely

In [ ]:
import os
import boto3
from google.colab import userdata


In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get('AWS_SECRET_ACCESS_KEY')

# AWS Bedrock Initialization Code Explanation

This document explains the two lines of Python code used to initialize a connection to Amazon Bedrock using the AWS SDK (`boto3`).

---

### Line 1: `client = boto3.client("bedrock-runtime", region_name="ap-south-1")`

#### **What it does:**
This line initializes a programmatic client session to interact with the **Amazon Bedrock Runtime API**.

#### **Breakdown:**
* **`boto3.client(...)`**: Tells Python to use the official AWS SDK (`boto3`) to create a connection service wrapper for a specific AWS service.
* **`"bedrock-runtime"`**: This specifies the exact service module to target. The `bedrock-runtime` service is optimized exclusively for low-latency, real-time data inference tasks like calling chat models or generating text. *(Note: This is different from the base `bedrock` service, which is used for administrative tasks like requesting model access or training models).*
* **`region_name="ap-south-1"`**: Configures the client to route all API calls directly to the **Asia Pacific (Mumbai)** data centre infrastructure. This reduces network latency if your application or notebook is physically running from India.

---

### Line 2: `model_Id = "apac.amazon.nova-lite-v1:0"`

#### **What it does:**
This line creates a configuration variable containing a unique identifier string that points to a specific foundational AI model version.

#### **Breakdown:**
* **`model_Id`**: A standard variable name used to pass into the `.converse()` or `.invoke_model()` functions later in your code script.
* **`"apac.amazon.nova-lite-v1:0"`**: This is a specific Amazon Bedrock **Cross-Region Inference Profile ID**.
  * **`apac`**: Stands for Asia-Pacific. It tells AWS to automatically route your request across multiple data centres within the Asia-Pacific territory to bypass traffic congestion or strict hardware limits.
  * **`amazon`**: Identifies Amazon as the creator/provider of the foundational model architecture.
  * **`nova-lite-v1:0`**: References version 1.0 of the **Amazon Nova Lite** model, which is a fast, highly cost-effective, multimodal model suited for processing quick text tasks and processing chat conversations.

---

### Key Takeaway for Your Notebook
When you execute these two lines together, your script prepares a stable, authenticated tunnel directly to the Mumbai region. It sets up your system to run inference on the **Amazon Nova Lite** model using smart Asia-Pacific routing, bypassing third-party marketplace billing validation rules entirely.


In [ ]:
client = boto3.client("bedrock-runtime", region_name="ap-south-1")
model_Id = "apac.amazon.nova-lite-v1:0"

In [ ]:
#build helper functions
def add_user_message(messages,text):
  user_message = {
      "role":"user",
      "content":[
          {"text":text}
      ]
  }
  messages.append(user_message)

def add_assistant_message(messages,text):
  assistant_message = {
      "role":"assistant",
      "content":[
          {"text":text}
      ]
  }
  messages.append(assistant_message)

def chat(messages):
  system_prompt = """
  You are an experienced AWS Support Specialist. Your job is to answer any queries regarding AWS Cloud Services
  """
  response = client.converse(
      modelId = model_Id,
      messages = messages,
      system = [{"text": system_prompt}]
  )
  return response["output"]["message"]["content"][0]["text"]

# Bedrock Chat Helper Functions Explanation

This document explains the purpose and mechanics of the three helper functions used to manage chat history and communicate with Amazon Bedrock.

---

### 1. `add_user_message(messages, text)`

#### **Purpose:**
Formats and appends a **user's input** into a standardized chat history list that Amazon Bedrock can understand.

#### **How it works:**
* It takes two arguments: the existing list of chat history (`messages`) and the new string text sent by the human (`text`).
* It creates a dictionary matching the strict structure required by the Bedrock Converse API:
  * `"role": "user"` specifies that a human sent this message.
  * `"content"` holds a list containing the text payload.
* It appends this formatted dictionary directly into your running `messages` list, updating your conversation history in place.

---

### 2. `add_assistant_message(messages, text)`

#### **Purpose:**
Formats and appends the **AI model's response** into the same chat history list to maintain context for multi-turn conversations.

#### **How it works:**
* It behaves almost identically to `add_user_message`, but with one crucial structural change: it sets `"role": "assistant"`.
* This tells Amazon Bedrock that this specific piece of text was generated by the AI model in a previous turn.
* Storing this correctly prevents the AI from getting confused about who said what when you ask follow-up questions.

---

### 3. `chat(messages)`

#### **Purpose:**
Sends the entire accumulated conversation history to Amazon Bedrock and extracts the final text response from the AI.

#### **How it works:**
* It takes your complete `messages` history list and passes it to the `client.converse()` method alongside your defined `model_Id`.
* Because you pass the whole list, the model can read previous turns, allowing it to remember the context of your conversation.
* The raw API response returned by AWS is a deeply nested dictionary. This function automatically digs through that dictionary layers-deep (`["output"]["message"]["content"][0]["text"]`) to cleanly pull out just the final text answer string and return it.

---

### Key Takeaway
Together, these three functions automate the tedious process of formatting JSON payloads. They allow you to build an interactive, multi-turn AI chatbot using clean, simple Python commands like `add_user_message(history, "Hello")` and `chat(history)`.


In [ ]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "whats 1+1?"
add_user_message(messages,"whats 1+1?")

# Pass the list of messages into chat to get an answer
answer = chat(messages)
answer

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# Add in the user's followup question
add_user_message(messages, "Add 3 more to that")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)
answer

'Certainly! If you add 3 more to the result of 1 + 1, which is 2, you perform the following addition:\n\n```\n  2\n+ 3\n-----\n  5\n```\n\nSo, 2 + 3 equals 5.'

In [ ]:
#this is the exercise function which asked us to develop a chat interface (multi-chat)
messages = []
while True:
  user_input = input("Enter Your Query (exit to close)==>  ")
  if user_input.lower() == "exit":
    break
  add_user_message(messages, user_input)
  answer = chat(messages)
  print(answer)
  add_assistant_message(messages,answer)
  print()


Enter Your Query (exit to close)==>  Hey hi, can you tell me about AWS EC2
Sure! Amazon Elastic Compute Cloud (EC2) is a web service that provides resizable compute capacity in the cloud. It allows you to quickly provision virtual servers, called instances, to run your applications. Here’s an overview of key features and components of AWS EC2:

### Key Features:

1. **Scalability**:
   - **Elasticity**: Easily scale your compute capacity up or down to match your application’s demand.
   - **Auto Scaling**: Automatically adjust the number of EC2 instances based on defined policies.

2. **Flexibility**:
   - **Wide Range of Instance Types**: Choose from various instance types optimized for different use cases, such as compute-optimized, memory-optimized, storage-optimized, and GPU instances.
   - **Predefined AMIs**: Use Amazon Machine Images (AMIs) to quickly launch instances with pre-configured software and settings.

3. **Security**:
   - **Network Security**: Use security groups and 